## Testing Our Fine Tuned Phi4-mini 3.8b Model locally

    Using ollama for the purpose

In [1]:
import json
import pandas as pd
from rank_bm25 import BM25Okapi
import ollama

    Some complex legal document to evaluate model

In [2]:
CONTRACT_TEXT = """
UNITED STATES SECURITIES AND EXCHANGE COMMISSION
WASHINGTON, D.C. 20549
EXHIBIT 10.1: MATERIAL CONTRACTS

*** TEXT OMITTED PURSUANT TO CONFIDENTIAL TREATMENT REQUEST ***
[DRAFT VERSION 9.4 - EXECUTION COPY - HIGHLY CONFIDENTIAL]

MASTER DEFINITIVE MERGER AND INTELLECTUAL PROPERTY CONSOLIDATION AGREEMENT

This Master Definitive Merger and Intellectual Property Consolidation Agreement (hereinafter referred to as the "Agreement", the "Merger Pact", or the "Consolidation Agreement") is entered into by and between Aegis Synthetics PLC, a public limited company formed under the laws of England and Wales with its principal place of business at 100 Bishopsgate, London ("Acquiring Party" or "Aegis"), Quantum Dynamix LLC, a Delaware limited liability company located at 400 Silicon Way, San Jose, California ("Target Entity" or "Quantum"), and Vanguard Holdings Group, a Cayman Islands exempted company acting as the administrative guarantor ("Guarantor").

WHEREAS, the Parties previously engaged in a Non-Disclosure Agreement dated March 4, 2025, and a non-binding Letter of Intent signed on October 12, 2025;
WHEREAS, Aegis desires to acquire all outstanding intellectual property assets, software source code, and hardware schematics owned by Quantum;
WHEREAS, the Guarantor has agreed to underwrite the financial escrow required to close this transaction;
NOW, THEREFORE, in consideration of the mutual covenants contained herein, and for other good and valuable consideration, the receipt and sufficiency of which are hereby acknowledged, the Parties agree to the following terms:

Article 1: Definitions and Interpretations.
1.1 "Licensed IP" means all patents, trademarks, trade secrets, and copyrighted material listed in Schedule C.
1.2 "Closing Date" means the date on which all financial transfers have cleared the Guarantor's escrow accounts.
1.3 "Key Personnel" refers to the C-suite executives of Quantum as defined in Appendix 2.

Article 2: Term and Effectiveness.
2.1 Signatures and Execution. The Parties acknowledge that physical signatures were captured on various dates ranging from December 15, 2026, to January 4, 2027. 
2.2 Commencement. Notwithstanding the signature dates or the provisional Letter of Intent, the substantive obligations, licensing transfers, and legal enforcement of this Agreement shall become legally binding and fully operational as of the 14th day of February, 2027 (the "Effective Date").

Article 3: Intellectual Property Transfer.
3.1 Upon the Effective Date, Quantum shall irrevocably transfer all Licensed IP to Aegis.
3.2 Quantum retains a limited, non-exclusive, non-transferable, royalty-free license to use the Licensed IP solely for internal research and development for a period of 24 months.

Article 4: Financial Compensation and Escrow.
4.1 The total purchase price is valued at $450,000,000 USD.
4.2 Aegis shall deposit an initial sum of $100,000,000 USD into the Guarantor's escrow account within 5 business days of the Effective Date.

Article 5 through Article 12: [Omitted for Brevity in Public Filing - Pertains to regulatory compliance, GDPR adherence, environmental safety standards, employee severance packages, and localized tax obligations in the UK and California].

Article 13: Term and Termination.
13.1 Termination for Material Breach. If any Party commits a material breach of its obligations under Article 3 or Article 4, the non-breaching Party must provide written notice detailing the breach. The breaching Party shall have forty-five (45) days to cure the defect. If uncured, the non-breaching Party may terminate this Agreement immediately.
13.2 Force Majeure. No Party shall be liable for failure to perform due to acts of God, global pandemics, or governmental embargoes.
13.3 Termination Without Cause. Despite the strict compliance rules outlined in Section 13.1, Aegis may elect to terminate this Agreement for convenience at any time, for any operational reason, provided that they deliver written notice of such termination to Quantum no less than ninety (90) days prior to the desired termination date.
13.4 Effect of Termination. Upon termination, all unpaid financial obligations become immediately void, and Quantum must destroy all Aegis-provided hardware.

Article 14: Indemnification and Liability Caps.
14.1 Quantum agrees to indemnify Aegis against any third-party claims asserting that the Licensed IP infringes upon existing patents.
14.2 Total liability for any damages arising under this Agreement shall be strictly capped at the total purchase price paid to date.

Article 15: Governing Law and Dispute Resolution.
15.1 Arbitration Mandate. All disputes must first go through binding arbitration in Geneva, Switzerland, under the WIPO Expedited Arbitration Rules.
15.2 Substantive Law. Regardless of the arbitration venue in Switzerland, the Parties explicitly reject the application of the United Nations Convention on Contracts for the International Sale of Goods. Instead, all contractual disputes, interpretations of clauses, and enforcement actions shall be governed by and construed in accordance with the domestic laws of the State of Delaware, explicitly excluding any conflict-of-law principles that would require the application of the laws of any other jurisdiction.

IN WITNESS WHEREOF, the duly authorized representatives of the Parties have executed this Master Definitive Merger and Intellectual Property Consolidation Agreement.
"""

In [3]:
def extract_abstract_data(abstract_text, model_name="secure-legal-abstractor-phi4-mini-q4"):
    """
    Takes an abstract, loops through the exact training prompt format, 
    and returns a clean dictionary of all extracted clauses.
    """
    target_columns = [
        "Document Name", 
        "Parties", 
        "Agreement Date", 
        "Effective Date", 
        "Expiration Date", 
        "Governing Law"
    ]
    
    extracted_data = {}

    print(f"[*] Processing abstract against {len(target_columns)} clauses...")

    for clause in target_columns:
        print(f"    -> Extracting: {clause}...")
        
        # This matches your fine-tuning EXACTLY. No raw JSON brackets to confuse it.
        messages = [
            {
                "role": "system", 
                "content": "You are a legal data extraction AI. Extract the requested entity from the text. If it is not present, output strictly 'Not found'."
            },
            {
                "role": "user", 
                "content": f"Extract the exact value for '{clause}'.\n\nContract Text:\n{abstract_text.strip()}"
            }
        ]

        try:
            response = ollama.chat(
                model=model_name,
                messages=messages,
                options={'temperature': 0.0} # Zero temperature for strict factual extraction
            )
            # Store the cleaned output
            extracted_data[clause] = response['message']['content'].strip()
            
        except Exception as e:
            extracted_data[clause] = f"Error: {str(e)}"

    return extracted_data

In [5]:
if __name__ == "__main__":
    
    sample_abstract = """
    This Software License and Maintenance Agreement is entered into on October 15, 2026, 
    by and between OpticMark Solutions LLC ("Licensor") and Nexus Global PLC ("Licensee"). 
    The Agreement shall become fully operational on November 1, 2026, and will remain in 
    effect until December 31, 2030, unless terminated earlier. This Agreement shall be 
    governed by the laws of the State of New York.
    """
    
    # Call the one function
    results = extract_abstract_data(CONTRACT_TEXT)
    
    # Print the beautiful, clean results
    print("\n================ FINAL OUTPUT ================")
    print(json.dumps(results, indent=4))

[*] Processing abstract against 6 clauses...
    -> Extracting: Document Name...
    -> Extracting: Parties...
    -> Extracting: Agreement Date...
    -> Extracting: Effective Date...
    -> Extracting: Expiration Date...
    -> Extracting: Governing Law...

================ FINAL OUTPUT ================
{
    "Document Name": "MAINTENANCE AND MAINTAINING AGREEMENT 1010-01/2021   [Omitted for Brevity in Public Filing]       Date:    Signed by Acquiring Party's representative on behalf ...         Target Entity... accelerated motion to dismiss, ...\n\n[PARTIES TO THIS MASTER DEFINITIVE MERGER AND INTELLECTUAL PROPERTY CONSOLIDATION AGREEMENT (THIS \"MAINTENANCE & MAINTAINING AGREEMENT\") are the identical parties as set forth in Article 1 of this Agreement. The following exhibits have been incorporated by reference and made a part hereof: ]",
    "Parties": "Acquiring Party: Aegis Synthetics PLC\nTarget Entity (Merger Target): Quantum Dynamix LLC (\"Quantum\")\nGuarantor: Vanguard Hold

- After too much testing it found out that 4-bit quantized model failing in finding right values.
- So after some testing of 16-bit in colab, the conclusion is our 16-bit model is capable of doing much better work compared to quantized one on gpu.